# OE2-A: reproducción autónoma en Google Colab (convergencia, latencia y throughput en proceso)

**Objetivo oficial (literal).** Determinar estadísticamente el desempeño propio de la propuesta basada en inteligencia de enjambre, considerando como indicadores el tiempo de convergencia, la latencia y el throughput bajo condiciones experimentales controladas.

**Qué comprueba este cuaderno.** Comprueba las dos mediciones del objetivo en un solo flujo: **componente A** (tiempo de convergencia, latencia y throughput en proceso, 5 000 ciclos) y **componente B** (latencia, throughput y tiempo de convergencia por HTTP, 120 corridas válidas). Para B ejecuta dos etapas en orden: B1, recálculo de los indicadores por corrida desde cada petición registrada; B2, análisis pre-registrado y exploratorio desde los indicadores por corrida. La conclusión final se calcula para ambos componentes.

**Datos de entrada.** Componente A: observaciones por ciclo (`raw_observations.csv`), tiempos de pared por lote, monitor y compuertas de reposo. Componente B: registros por petición, `runs.jsonl` (136 registros finales), `intentos.jsonl`, monitores de los dos segmentos y planes. Scripts de análisis sellados y resultados locales para comparar.

**Qué no hace.** No recalcula los tiempos medidos, que son los del equipo local con interferencia registrada. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Pulse «Ejecutar todo» en un entorno nuevo de Colab: los datos se descargan solos desde GitHub (commit fijado) y se verifican por SHA-256; no hay que subir ni buscar archivos.

**Alcance de este cuaderno.** Solo el componente A del OE2 (medición en proceso, 5 000 ciclos). El componente B (HTTP) está en `OE2B_reproduccion_autonoma.ipynb`.

**Contenido.** 1 descripción · 2 entorno y código (clonado de la etiqueta inmutable) · 3 descarga automática y verificación SHA-256 · 4 validación de integridad y estructura · 5 preprocesamiento y cálculo de indicadores (scripts sellados) · 6 análisis estadístico · 7 comparación con el resultado local y con las cifras de la tesis · 8 tablas · 9 gráficos · 10 pruebas automáticas · 11 resumen. Las diferencias se clasifican como exacta, dentro de tolerancia, explicada por versión, no explicada o no reproducible.

## 1. Entorno y código de análisis
El código vive en el repositorio público (módulos legibles en `reproducibility/etapa26_colab_autonomo/src/repro/`); los scripts de análisis sellados vienen dentro de los paquetes de datos, con su hash verificado.

In [ ]:
import os, sys, platform, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Renato5Lara/multiagent-llm-education.git'
REPO_REF = 'repro-etapa26-v2'            # etiqueta inmutable del codigo; los datos estan fijados por commit en manifests/data_manifest.json
if os.environ.get('REPRO_LOCAL_REPO'):   # solo para pruebas locales antes de publicar la etiqueta
    REPO_DIR = os.environ['REPRO_LOCAL_REPO']
else:
    REPO_DIR = 'repo_repro'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch', REPO_REF, '--filter=blob:none', '--sparse', REPO_URL, REPO_DIR], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'sparse-checkout', 'set', 'reproducibility/etapa26_colab_autonomo'], check=True)   # solo el codigo de esta etapa
sys.path.insert(0, os.path.join(REPO_DIR, 'reproducibility', 'etapa26_colab_autonomo', 'src'))
import repro
from repro import sh
print('Codigo de analisis:', REPO_URL, '@', REPO_REF)


In [ ]:
try:
    import statsmodels
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'statsmodels'], check=True)
    import statsmodels
import statsmodels, numpy, scipy, pandas, matplotlib
ENTORNO = dict(python=platform.python_version(), plataforma=platform.platform(), numpy=numpy.__version__, scipy=scipy.__version__, pandas=pandas.__version__, matplotlib=matplotlib.__version__, cpu=os.cpu_count(), statsmodels=statsmodels.__version__)
print(ENTORNO)


## 2. Descarga automática y verificación de los paquetes (SHA-256)

In [ ]:
# Descarga automatica desde GitHub (commit fijado) y verificacion SHA-256 contra el manifiesto. No hay que subir nada.
# Si falta un archivo, no se puede descargar o su hash no coincide, se detiene con el archivo, la URL y el hash esperado.
PAQUETES = ['oe2a_colab_paquete']
repro.fetch_packages(PAQUETES, dest='.')


## 3. Verificación interna, cálculo y análisis (celdas del notebook de comprobación publicado, sin cambios salvo la ejecución de scripts)

In [ ]:
import hashlib
base='oe2a/campana_01'; bad=[]; checked=[]
for l in open(base+'/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,n=l.split(None,1); n=n.strip().lstrip('*'); p=os.path.join(base,n)
    if os.path.exists(p):
        checked.append(n)
        if hashlib.sha256(open(p,'rb').read()).hexdigest()!=h: bad.append(n)
print('verificados:', checked); print('hash distinto:', bad); assert not bad

In [ ]:
sh(f"python oe2a/analisis_oe2a.py oe2a/campana_01/raw_observations.csv oe2a/campana_01/lotes_tiempo_pared.json oe2a/salida_colab oe2a/campana_01_monitor.csv oe2a/gate_inicio_campana01.json oe2a/gate_cierre_campana01.json > oe2a/log_analisis.txt 2>&1")
print(open('oe2a/log_analisis.txt').read()[-400:])

In [ ]:
import json, math
a=json.load(open('oe2a/salida_colab/resultados_oe2a.json')); b=json.load(open('oe2a/esperado_resultados_oe2a_local.json'))
def hoja(x, p=''):
    if isinstance(x, dict):
        for k,v in x.items(): yield from hoja(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i,v in enumerate(x): yield from hoja(v, p+'/'+str(i))
    else: yield p, x
A=dict(hoja(a)); B=dict(hoja(b))
print('mismas claves:', set(A)==set(B), '| hojas:', len(A))
mx=0; peor=None; dist=[]
for k in A:
    va, vb = A[k], B.get(k)
    if isinstance(va,(int,float)) and isinstance(vb,(int,float)) and not isinstance(va,bool):
        dif=abs(va-vb); rel=dif/max(abs(vb),1e-12)
        if dif>mx: mx=dif; peor=(k,va,vb)
        if rel>1e-9 and dif>1e-9: dist.append((k,va,vb))
    elif va!=vb: dist.append((k,va,vb))
print('diferencia absoluta maxima:', mx, peor)
print('hojas distintas (rel>1e-9):', len(dist)); print(dist[:15])
print('T_conv media geometrica (ms): Colab', a['tconv_I02']['media_geometrica_ms'], '| local', b['tconv_I02']['media_geometrica_ms'])
print('IC95 t:', a['tconv_I02']['ic95_t_ms'], b['tconv_I02']['ic95_t_ms'])
print('IC95 bootstrap:', a['tconv_I02']['ic95_bootstrap_percentil_ms'], b['tconv_I02']['ic95_bootstrap_percentil_ms'])
print('P95 latencia en proceso:', a['latencia_en_proceso']['percentiles_ms_ciclos']['95'], b['latencia_en_proceso']['percentiles_ms_ciclos']['95'])
print('X_proc entre lotes:', a['throughput_en_proceso']['entre_lotes']['media'], b['throughput_en_proceso']['entre_lotes']['media'])
print('CR:', a['tasa_convergencia_I03']['CR'], a['tasa_convergencia_I03']['ic95_wilson'])
print('instrumentacion cumple:', a['instrumentacion']['cumple'], b['instrumentacion']['cumple'])

## 4. Comparación con las cifras de la tesis
Cada cifra se lee del resultado **reproducido** y se compara con la que figura en la tesis (versión CIERRE_FINAL) y en el JSON oficial sellado (`manifests/valores_oficiales.json`).

In [ ]:
from repro import official, validation, tables, figures
from repro.data_loading import sha256_file
import pandas as pd
RUTAS = {'resultados_oe2a.json': 'oe2a/salida_colab/resultados_oe2a.json'}
cifras = official.evaluar('OE2-A', RUTAS)
print(cifras[['id', 'descripcion', 'esperado', 'obtenido', 'dif', 'estado']].to_string(index=False))


## 5. Clasificación de las diferencias frente al resultado local
Exacta · dentro de tolerancia (1e-9) · explicada por versión · no explicada · no reproducible.

In [ ]:
import scipy
REG = validation.Registro(scipy_version=scipy.__version__)
COMPARACIONES = [('OE2-A resultados_oe2a.json', 'oe2a/salida_colab/resultados_oe2a.json', 'oe2a/esperado_resultados_oe2a_local.json')]
for _e, _a, _b in COMPARACIONES:
    REG.comparar(_e, _a, _b)


## 6. Tablas

In [ ]:
OUT = 'outputs'
TABLAS = {'medias_por_perfil': 'oe2a/salida_colab/medias_por_perfil.csv'}
tablas_generadas = tables.exportar_tablas('OE2-A', TABLAS, os.path.join(OUT, 'tables'))
print('Tablas generadas:')
for _t in tablas_generadas:
    print('  ', _t)


## 7. Gráficos
Se generan desde la salida del análisis ejecutado arriba (no desde valores copiados). Se exportan PNG (300 ppi), SVG y el CSV de la serie dibujada (`figure_data/`).

In [ ]:
METAS = official.cargar_valores()['metas_de_diseno']
fig = figures.fig_oe2a_complementaria('oe2a/campana_01/raw_observations.csv', RUTAS['resultados_oe2a.json'], os.path.join(OUT, 'figures', 'OE2-A'))
print('Figura COMPLEMENTARIA (no figura en la tesis): ' + fig['png'])
try:
    from IPython.display import Image, display
    display(Image(fig['png']))
except Exception:
    pass


## 8. Pruebas automáticas de consistencia

In [ ]:
fallos = []
if cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE']).any():
    fallos.append('cifras oficiales: ' + str(cifras[cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE'])][['id', 'estado']].values.tolist()))
peor, cuenta = REG.veredicto()
if peor in ('NO_EXPLICADA', 'NO_REPRODUCIBLE'):
    fallos.append('comparacion con el resultado local: ' + peor)
if not os.path.getsize(fig['png']) > 0:
    fallos.append('figura vacia')
assert not fallos, fallos
print('PRUEBAS AUTOMATICAS DE CONSISTENCIA: todas superadas | cifras oficiales comprobadas:', len(cifras), '| comparaciones:', cuenta)


## 9. Resumen final de la reproducción

In [ ]:
import shutil
resumen = tables.guardar_resumen('OE2-A', os.path.join(OUT, 'reports'), REG.tabla(), cifras, ENTORNO, extra=dict(veredicto=peor, conteo=cuenta, paquetes=PAQUETES))
print('RESUMEN DE REPRODUCCION OE2-A')
print(REG.tabla()[['comparacion', 'categoria', 'dif_abs_max']].to_string(index=False))
print('Veredicto (peor categoria):', peor, cuenta)
print('Cifras de la tesis comprobadas: %d (%d coinciden exactamente, %d dentro del redondeo)' % (len(cifras), (cifras.estado == 'COINCIDE').sum(), (cifras.estado == 'COINCIDE_REDONDEO').sum()))
print('LIMITACIONES: 5 000 ciclos en un solo flujo (no mide capacidad); 2 770 ciclos con interferencia fuerte; la campana OE2A-02 (replica controlada) no se ejecuto por cuatro fallos de la compuerta de inicio.')
print('Alcance: se reproduce el analisis, no la adquisicion de datos; no se repite ninguna campana.')
_z = shutil.make_archive('resultados_OE2-A', 'zip', OUT)
print('Salidas empaquetadas en', os.path.basename(_z))
if IN_COLAB:
    from google.colab import files
    files.download(_z)
